# LSST Science Pipelines — Installation Check

This notebook verifies that the LSST Science Pipelines are installed, reports the version, and runs a simple demonstration using core pipeline tools.

In [1]:
import sys
import subprocess
import numpy as np

## 1. Check installation

In [2]:
# Try to import lsst package and see if it succeeds
try:
    import lsst.utils
    import lsst.geom as geom
    import lsst.afw.table as afwTable
    print("LSST Science Pipelines are successfullyinstalled.")
except ImportError as e:
    print(f"LSST Science Pipelines not found: {e}")

LSST Science Pipelines are successfullyinstalled.


## 2. Get the version of thr pipelines 

Check it is the same as the version passed to Docker

In [3]:
# eups is the package manager used by LSST — query the lsst_distrib version
result = subprocess.run(
    ["eups", "list", "-s", "lsst_distrib"],
    capture_output=True, text=True
)

if result.returncode == 0 and result.stdout.strip():
    print(f"lsst_distrib version: {result.stdout.strip()}")

lsst_distrib version: gc675d380bf+01ded468f7 	v29_2_1 current setup


In [4]:
# Also show the Python environment details
print(f"Python:  {sys.version}")
print(f"Prefix:  {sys.prefix}")

Python:  3.12.11 | packaged by conda-forge | (main, Jun  4 2025, 14:34:08) [GCC 13.3.0]
Prefix:  /opt/lsst/software/stack/conda/envs/lsst-scipipe-10.1.0


## 3. A simple demo — create and manipulate a `lsst.geom.SpherePoint`

We use `lsst.geom` to create a sky coordinate and compute the angular separation between two points — a fundamental operation in survey astronomy.

In [5]:
# Define two sky coordinates (RA, Dec) in degrees roughly the COSMOS field
ra1, dec1 = 150.0, 2.0   
ra2, dec2 = 150.1, 2.1

point1 = geom.SpherePoint(ra1 * geom.degrees, dec1 * geom.degrees)
point2 = geom.SpherePoint(ra2 * geom.degrees, dec2 * geom.degrees)

separation = point1.separation(point2)

print(f"Point 1:     RA={point1.getRa().asDegrees():.4f}°, Dec={point1.getDec().asDegrees():.4f}°")
print(f"Point 2:     RA={point2.getRa().asDegrees():.4f}°, Dec={point2.getDec().asDegrees():.4f}°")
print(f"Separation:  {separation.asArcminutes():.4f} arcmin  ({separation.asDegrees():.6f}°)")

Point 1:     RA=150.0000°, Dec=2.0000°
Point 2:     RA=150.1000°, Dec=2.1000°
Separation:  8.4826 arcmin  (0.141376°)


## 4. A simple demo — build a small `lsst.afw.table.SourceCatalog`

`SourceTable.makeMinimalSchema()` is the correct starting point for a `SourceCatalog`, but it is
**not** a blank slate — it mandates four fields that the pipeline always expects to be populated:
`id`, `coord_ra`, `coord_dec`, and `parent`.  Leaving them at their default zero values would
produce a catalog where every source appears to sit at RA=0°, Dec=0° with no parent information,
which misrepresents what the pipeline actually stores.  Below we extend the schema with custom
fields and then populate **all** required fields per record, including a realistic sky coordinate
and a `parent` of 0 (meaning the source is unblended).

In [6]:
# Create a minimal schema with default fields
schema = afwTable.SourceTable.makeMinimalSchema()
print(schema)

Schema(
    (Field['L'](name="id", doc="unique ID"), Key<L>(offset=0, nElements=1)),
    (Field['Angle'](name="coord_ra", doc="position in ra/dec"), Key<Angle>(offset=8, nElements=1)),
    (Field['Angle'](name="coord_dec", doc="position in ra/dec"), Key<Angle>(offset=16, nElements=1)),
    (Field['L'](name="parent", doc="unique ID of parent source"), Key<L>(offset=24, nElements=1)),
)



In [7]:
# Add some additional fields
fluxKey  = schema.addField("flux",  type=float, doc="Source flux (nJy)", units="nJy")
labelKey = schema.addField("label", type=str,   doc="Source label", size=20)
print(schema.getNames())

{'flux', 'label', 'parent', 'id', 'coord_dec', 'coord_ra'}


In [8]:
# Create a catalog with the dedined schema
catalog = afwTable.SourceCatalog(schema)
catalog.schema

Schema(
    (Field['L'](name="id", doc="unique ID"), Key<L>(offset=0, nElements=1)),
    (Field['Angle'](name="coord_ra", doc="position in ra/dec"), Key<Angle>(offset=8, nElements=1)),
    (Field['Angle'](name="coord_dec", doc="position in ra/dec"), Key<Angle>(offset=16, nElements=1)),
    (Field['L'](name="parent", doc="unique ID of parent source"), Key<L>(offset=24, nElements=1)),
    (Field['D'](name="flux", doc="Source flux (nJy)", units="nJy"), Key<D>(offset=32, nElements=1)),
    (Field['String'](name="label", doc="Source label", size=20), Key<String>(offset=40, nElements=20)),
)

In [9]:
# Populate the table with some entries (sources)
# Flux values are realistic for Rubin/LSST:
#   ~10 000 nJy  ≈ r=22 mag  (bright star / galaxy)
#   ~  400 nJy   ≈ r=25 mag  (faint galaxy near single-visit limit)
#   ~   15 nJy   ≈ r=28 mag  (deep-stack detection, close to 5σ limit)
# parent=0 means the source is unblended (no parent detection).
sources = [
    (1, 150.01, 2.05, 10234.5, "star"),
    (2, 150.02, 2.08,   412.3, "galaxy"),
    (3, 150.00, 2.12,    14.7, "artifact"),
]

parentKey = schema["parent"].asKey()

# Create a new record in the catalog for each source 
for sid, ra, dec, flux_nJy, label in sources:
    record = catalog.addNew()
    record.setId(sid)
    record.setCoord(geom.SpherePoint(ra * geom.degrees, dec * geom.degrees))
    record.set(parentKey, 0)
    record.set(fluxKey, flux_nJy)
    record.set(labelKey, label)

In [10]:
# Check the catalog entries
print(f"Catalog contains {len(catalog)} sources\n")
print(f"{'ID':>4}  {'RA (deg)':>10}  {'Dec (deg)':>10}  {'Flux (nJy)':>12}  {'Label'}")
print("-" * 62)
for rec in catalog:
    coord = rec.getCoord()
    print(
        f"{rec.getId():>4}  "
        f"{coord.getRa().asDegrees():>10.4f}  "
        f"{coord.getDec().asDegrees():>10.4f}  "
        f"{rec.get(fluxKey):>12.2f}  "
        f"{rec.get(labelKey)}"
    )

Catalog contains 3 sources

  ID    RA (deg)   Dec (deg)    Flux (nJy)  Label
--------------------------------------------------------------
   1    150.0100      2.0500      10234.50  star
   2    150.0200      2.0800        412.30  galaxy
   3    150.0000      2.1200         14.70  artifact


If all cells above ran without errors, your LSST Science Pipelines installation is working correctly.